# Colab GPU runtime check (A100, G4, L4, T4)
Run on each GPU runtime in turn and copy the printed JSON back. The key fields are the GPU name and memory, the driver version, and whether the driver supports CUDA 13, which vLLM 0.33 needs.

In [ ]:
import subprocess, sys, shutil, os, platform, json, time
def sh(c):
    try: return subprocess.run(c, shell=True, capture_output=True, text=True, timeout=90).stdout.strip()
    except Exception as e: return f"ERR {e}"
info = {
  "time": time.strftime("%Y-%m-%dT%H:%M"),
  "gpu": sh("nvidia-smi --query-gpu=name,memory.total,power.limit,driver_version,compute_cap --format=csv,noheader"),
  "cuda_driver": sh("nvidia-smi | grep -o 'CUDA Version: [0-9.]*'"),
  "python": sys.version.split()[0],
  "torch": sh("python -c 'import torch;print(torch.__version__, torch.version.cuda)'"),
  "cpu_cores": os.cpu_count(),
  "ram_gb": round(os.sysconf('SC_PAGE_SIZE')*os.sysconf('SC_PHYS_PAGES')/1e9, 1),
  "disk_free_gb": round(shutil.disk_usage('/content').free/1e9, 1),
  "hf_reachable": sh("curl -s -o /dev/null -w '%{http_code}' https://huggingface.co"),
}
# Does the driver support CUDA 13 (needed by vLLM 0.33 wheels)? Driver 580 or newer.
drv = info["gpu"].split(",")[3].strip() if info["gpu"].count(",") >= 3 else ""
info["cuda13_capable_driver"] = (int(drv.split(".")[0]) >= 580) if drv[:1].isdigit() else None
print(json.dumps(info, indent=1))

In [ ]:
# Optional: does vLLM 0.33 install and start here? (about 5 minutes; skip on T4 if short of time)
!pip -q install "vllm==0.33.0" 2>&1 | tail -3
!python -c "import vllm, torch; print('vllm', vllm.__version__, 'torch', torch.__version__, torch.cuda.is_available())"